In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
from pyspark.sql.functions import desc, sum, count, avg, col
df = spark.read.parquet(f"{presentation_folder_path}/race_results")
display(df)

In [0]:
best_drivers_df = (
    df.groupBy("driver_name")
    .agg(
        count("*").alias("race_count"),
        sum("points").alias("total_points"),
        avg("points").alias("average_points"),
        avg("position").alias("average_position")
    )
    .filter(col("race_count") > 50)
    .orderBy(desc("average_points"))
)

display(best_drivers_df)

In [0]:
best_nationality_df = (
    df.groupBy("driver_nationality")
    .agg(
        count("*").alias("race_count"),
        sum("points").alias("total_points"),
        avg("points").alias("average_points"),
        avg("position").alias("average_position")
    )
    .filter(col("race_count") > 200)
    .orderBy(desc("average_points"))
)

display(best_nationality_df)

In [0]:
import plotly.express as px
import plotly.express as px

# 1. Limit the data to the Top 15 drivers BEFORE plotting
# Since your PySpark DataFrame is already sorted by average_points descending, 
# taking the first 15 rows gives us the best drivers.
pdf = best_drivers_df.limit(15).toPandas()

# 2. Create the enhanced interactive bar chart
fig = px.bar(
    pdf, 
    x="driver_name", 
    y="average_points", 
    color="race_count", 
    text="average_points", # Displays the exact points on each bar
    color_continuous_scale="Plasma", # A much nicer, brighter color gradient
    title="Top 15 Nationalities by Average Points (Min 200 Races)",
    labels={
        "driver_nationality": "Nationality",
        "average_points": "Avg Points/Race",
        "race_count": "Total Races",
    }
)

# 3. Fine-tune the layout and formatting
fig.update_traces(
    texttemplate='%{text:.2f}', # Formats the bar labels to 2 decimal places
    textposition='outside'      # Pushes the labels above the bars
)

fig.update_layout(
    xaxis_tickangle=-45, # Tilts the driver names so they don't overlap
    height=600,          # Makes the chart a bit taller
    margin=dict(t=80, b=80, l=40, r=40) # Adds breathing room around the edges
)

# 4. Display the chart
fig.show()

In [0]:
import plotly.express as px
import plotly.express as px

# 1. Limit the data to the Top 15 drivers BEFORE plotting
# Since your PySpark DataFrame is already sorted by average_points descending, 
# taking the first 15 rows gives us the best drivers.
pdf = best_nationality_df.limit(15).toPandas()

# 2. Create the enhanced interactive bar chart
fig = px.bar(
    pdf, 
    x="driver_nationality", 
    y="average_points", 
    color="race_count", 
    text="average_points", # Displays the exact points on each bar
    color_continuous_scale="Plasma", # A much nicer, brighter color gradient
    title="Top 15 Drivers by Average Points (Min 50 Races)",
    labels={
        "driver_nationality": "Driver", 
        "average_points": "Avg Points/Race", 
        "race_count": "Total Races"
    }
)

# 3. Fine-tune the layout and formatting
fig.update_traces(
    texttemplate='%{text:.2f}', # Formats the bar labels to 2 decimal places
    textposition='outside'      # Pushes the labels above the bars
)

fig.update_layout(
    xaxis_tickangle=-45, # Tilts the driver names so they don't overlap
    height=600,          # Makes the chart a bit taller
    margin=dict(t=80, b=80, l=40, r=40) # Adds breathing room around the edges
)

# 4. Display the chart
fig.show()

In [0]:
%sh
sudo apt-get update
sudo apt-get install -y ffmpeg

In [0]:
import re
import unicodedata

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

START_YEAR = 1950
END_YEAR = 2025

round_col = next((c for c in ["race_round", "round", "round_number"] if c in df.columns), None)

order_cols = [F.col("race_year").asc()]
if round_col:
    order_cols.append(F.col(round_col).asc_nulls_last())
order_cols.append(F.col("race_date").asc_nulls_last())

# Keep raw points only in Spark — we RECOMPUTE career_points in pandas after dates are fixed
df_historical = (
    df.withColumn("points", F.col("points").cast("double"))
      .filter((F.col("race_year") >= START_YEAR) & (F.col("race_year") <= END_YEAR))
)

pdf_career = df_historical.toPandas()

def norm_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", s).strip()

pdf_career["driver_name"] = pdf_career["driver_name"].map(norm_name)
pdf_career["race_date"] = pd.to_datetime(pdf_career["race_date"], errors="coerce")
pdf_career["race_year"] = pd.to_numeric(pdf_career["race_year"], errors="coerce").astype("Int64")
pdf_career["points"] = pd.to_numeric(pdf_career["points"], errors="coerce").fillna(0.0)

# One shared calendar date per race (not per driver-row)
race_key_cols = ["race_year"]
if round_col and round_col in pdf_career.columns:
    race_key_cols.append(round_col)
elif "race_name" in pdf_career.columns:
    race_key_cols.append("race_name")
elif "circuit_name" in pdf_career.columns:
    race_key_cols.append("circuit_name")
else:
    pdf_career["_fallback_race_key"] = pdf_career["race_date"].astype(str)
    race_key_cols.append("_fallback_race_key")

race_calendar = (
    pdf_career.groupby(race_key_cols, dropna=False)["race_date"]
    .agg(real_date=lambda s: s.dropna().min() if s.notna().any() else pd.NaT)
    .reset_index()
)

sort_cols = ["race_year"] + ([round_col] if round_col and round_col in race_calendar.columns else [])
race_calendar = race_calendar.sort_values(sort_cols).reset_index(drop=True)
race_calendar["race_order"] = race_calendar.groupby("race_year").cumcount()
race_calendar["total_races"] = (
    race_calendar.groupby("race_year")["race_year"].transform("count").clip(lower=1)
)

synth = (
    pd.to_datetime(race_calendar["race_year"].astype(str) + "-03-15")
    + pd.to_timedelta(
        (race_calendar["race_order"] / race_calendar["total_races"]) * 240.0,
        unit="D",
    )
)
race_calendar["race_date_fixed"] = race_calendar["real_date"].fillna(synth)

pdf_career = pdf_career.merge(
    race_calendar[race_key_cols + ["race_date_fixed"]],
    on=race_key_cols,
    how="left",
)
pdf_career["race_date"] = pdf_career["race_date_fixed"]
pdf_career = pdf_career.drop(columns=["race_date_fixed", "_fallback_race_key"], errors="ignore")
pdf_career = pdf_career.dropna(subset=["race_date", "driver_name", "race_year"])

# Collapse duplicate driver/date rows, THEN cumulative sum in final date order
# (this removes the sawtooth from date reshuffling)
pdf_career = (
    pdf_career.groupby(["driver_name", "race_date"], as_index=False)["points"]
    .sum()
    .sort_values(["driver_name", "race_date"])
    .reset_index(drop=True)
)
pdf_career["career_points"] = pdf_career.groupby("driver_name")["points"].cumsum()

# Hard guarantee: cumulative series never goes backwards
pdf_career["career_points"] = pdf_career.groupby("driver_name")["career_points"].cummax()

assert pdf_career["race_date"].isna().sum() == 0
assert (
    pdf_career.groupby("driver_name")["career_points"].diff().fillna(0) >= -1e-9
).all(), "career_points still not monotonic"

print(
    f"rows={len(pdf_career):,} | drivers={pdf_career['driver_name'].nunique()} | "
    f"unique_dates={pdf_career['race_date'].nunique()} | "
    f"{pdf_career['race_date'].min().date()} → {pdf_career['race_date'].max().date()}"
)

In [0]:
import base64
import re
import unicodedata

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.animation import FuncAnimation

# -------- pacing --------
FULL_TIMELINE = True
PREVIEW_LAST_YEARS = 8
DURATION_SEC = 55          # total video length — raise for slower
FPS = 20
DPI = 120
OUT_PATH = "/tmp/f1_all_time_best.mp4"

BG = "#0B0D10"
GRID = "#2A2E36"
MUTED = "#8B93A7"
WHITE = "#F4F5F7"

DRIVER_COLORS = {
    "Lewis Hamilton": "#00D2BE",
    "Sebastian Vettel": "#FF1801",
    "Fernando Alonso": "#00FF66",
    "Kimi Raikkonen": "#FF007F",
    "Nico Rosberg": "#E0E0E0",
    "Max Verstappen": "#2A75FF",
    "Michael Schumacher": "#FF1801",
    "Ayrton Senna": "#FFD700",
    "Alain Prost": "#FFB800",
    "Niki Lauda": "#C4C4C4",
    "Jackie Stewart": "#0033A0",
    "Nelson Piquet": "#60A5FA",
    "Nigel Mansell": "#3B82F6",
    "Jenson Button": "#00A0DE",
    "Daniel Ricciardo": "#FFB800",
    "Valtteri Bottas": "#00D2BE",
}

def norm_name(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", s).strip()

def driver_color(name: str) -> str:
    name = norm_name(name).title()
    if name in DRIVER_COLORS:
        return DRIVER_COLORS[name]
    palette = ["#F59E0B", "#A78BFA", "#34D399", "#F472B6", "#60A5FA", "#F87171"]
    return palette[hash(name) % len(palette)]

def adjust_labels(label_data, min_dist):
    label_data = sorted(label_data, key=lambda x: x["y_val"])
    for i in range(1, len(label_data)):
        if label_data[i]["y_val"] - label_data[i - 1]["y_val"] < min_dist:
            label_data[i]["y_val"] = label_data[i - 1]["y_val"] + min_dist
    return label_data

pdf = pdf_career.copy()
pdf["driver_name"] = pdf["driver_name"].map(norm_name)
pdf["race_date"] = pd.to_datetime(pdf["race_date"], errors="coerce")
pdf = pdf.dropna(subset=["race_date", "career_points", "driver_name"])
pdf = pdf.sort_values(["driver_name", "race_date"])

# Extra safety if Notebook 1 wasn't re-run yet
pdf["career_points"] = pdf.groupby("driver_name")["career_points"].cummax()

# -------- build smooth interpolated series per driver --------
if FULL_TIMELINE:
    t0, t1 = pdf["race_date"].min(), pdf["race_date"].max()
else:
    t1 = pdf["race_date"].max()
    t0 = t1 - pd.DateOffset(years=PREVIEW_LAST_YEARS)

n_frames = max(int(DURATION_SEC * FPS), 60)
frame_dates = pd.date_range(t0, t1, periods=n_frames)

# Precompute interpolated career points on the shared timeline
driver_curves = {}
for driver, g in pdf.groupby("driver_name", sort=False):
    g = g.sort_values("race_date")
    x = g["race_date"].map(pd.Timestamp.toordinal).to_numpy(dtype=float)
    y = g["career_points"].to_numpy(dtype=float)
    # dedupe ordinals (keep last = highest cum points that day)
    if len(x) > 1:
        _, uniq_idx = np.unique(x, return_index=True)
        # unique returns first occurrence — we want last
        mask = np.ones(len(x), dtype=bool)
        mask[:-1] = x[1:] != x[:-1]
        x, y = x[mask], y[mask]

    xi = frame_dates.map(pd.Timestamp.toordinal).to_numpy(dtype=float)
    # flat before first race, interpolate between races, flat after last
    yi = np.interp(xi, x, y, left=np.nan, right=y[-1])
    driver_curves[driver] = yi

driver_names = list(driver_curves.keys())
curve_matrix = np.vstack([driver_curves[d] for d in driver_names])  # shape: drivers x frames

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": WHITE,
    "axes.labelcolor": MUTED,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
})

fig, ax = plt.subplots(figsize=(14, 7), dpi=DPI, facecolor=BG)
fig.patch.set_facecolor(BG)
fig.subplots_adjust(left=0.07, right=0.82, top=0.88, bottom=0.10)

def update(frame_idx):
    ax.clear()
    ax.set_facecolor(BG)
    fig.patch.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.spines["left"].set_color(GRID)
    ax.spines["bottom"].set_color(GRID)

    current_date = frame_dates[frame_idx]
    vals = curve_matrix[:, frame_idx]
    valid = np.isfinite(vals)
    if not valid.any():
        return

    standings_idx = np.where(valid)[0]
    top5_local = standings_idx[np.argsort(vals[standings_idx])[::-1][:5]]
    max_y = float(np.nanmax(vals))
    if not np.isfinite(max_y) or max_y <= 0:
        return

    labels = []
    x_hist = frame_dates[: frame_idx + 1]

    for di in top5_local:
        driver = driver_names[di]
        y_hist = curve_matrix[di, : frame_idx + 1]
        mask = np.isfinite(y_hist)
        if mask.sum() < 1:
            continue

        color = driver_color(driver)
        last_points = float(y_hist[mask][-1])

        ax.plot(x_hist[mask], y_hist[mask], lw=8, color=color, alpha=0.18, solid_capstyle="round")
        ax.plot(x_hist[mask], y_hist[mask], lw=2.8, color=color, alpha=0.95, solid_capstyle="round")
        ax.scatter(
            [current_date], [last_points],
            s=55, color=color, edgecolors=WHITE, linewidths=1.2, zorder=5,
        )
        labels.append({
            "y_val": last_points,
            "text": f"{driver}  {int(round(last_points)):,}",
            "color": color,
        })

    labels = adjust_labels(labels, min_dist=max_y * 0.055)
    for lbl in labels:
        ax.text(
            current_date + pd.Timedelta(days=20),
            lbl["y_val"],
            lbl["text"],
            color=lbl["color"],
            fontsize=11,
            fontweight="bold",
            va="center",
            ha="left",
            clip_on=False,
        )

    window_start = max(pd.Timestamp(t0), current_date - pd.DateOffset(years=5))
    window_end = current_date + pd.DateOffset(months=20)
    ax.set_xlim(window_start, window_end)
    ax.set_ylim(0, max_y * 1.14)

    ax.set_title(
        f"ALL-TIME BEST F1 DRIVERS   ·   {current_date.year}",
        fontsize=18, fontweight="bold", color=WHITE, pad=16, loc="left",
    )
    ax.set_ylabel("Total career points", fontsize=11, color=MUTED)
    ax.grid(True, axis="y", color=GRID, linestyle="-", linewidth=0.8, alpha=0.85)
    ax.grid(False, axis="x")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"{int(x):,}"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(colors=MUTED, labelsize=10, length=0)
    ax.set_axisbelow(True)

anim = FuncAnimation(fig, update, frames=n_frames, interval=1000 / FPS)

anim.save(
    OUT_PATH,
    writer="ffmpeg",
    fps=FPS,
    dpi=DPI,
    savefig_kwargs={"facecolor": BG, "edgecolor": "none"},
)
plt.close(fig)

with open(OUT_PATH, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")

displayHTML(f"""
<video width="960" height="540" controls autoplay
  style="border-radius:10px; box-shadow:0 12px 40px rgba(0,0,0,.55); background:#0B0D10;">
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
""")